In [ ]:
!git clone https://github.com/rohban-lab/GhostWord.git

%cd GhostWord/demo

In [ ]:
import os
import json
import torch
import shutil
import random
import librosa
import torchaudio
import numpy as np
import pandas as pd
from tqdm import tqdm
import soundfile as sf
from pathlib import Path
from scipy.signal import butter, lfilter
from torch.utils.data import DataLoader
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, TrainerCallback

In [ ]:
# -----------------------------------------------------------------------------------------------
def set_global_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    

set_global_seed(123)

In [ ]:
# -----------------------------------------------------------------------------------------------
def bandpass_filter(data, sr, lowcut, highcut):
    nyq = 0.5 * sr
    low = lowcut / nyq
    high = highcut / nyq
    b, a = butter(4, [low, high], btype="band")
    return lfilter(b, a, data)


# -----------------------------------------------------------------------------------------------
def generate_noise(sr=16000, duration=0.5, noise_type="white", freq_range=None, seed=None):
    np.random.seed(seed)
    samples = int(duration * sr)
    white = np.random.normal(0, 1, samples)

    if noise_type == "white":
        noise = white
    elif noise_type == "pink":
        # Simple pink approximation (1/f filter)
        freqs = np.fft.rfftfreq(samples, 1/sr)
        spectrum = np.fft.rfft(white)
        spectrum /= np.sqrt(np.maximum(freqs, 1))
        noise = np.fft.irfft(spectrum)
    elif noise_type == "band":
        if freq_range is None:
            freq_range = (500, 1000)
        noise = bandpass_filter(white, sr, freq_range[0], freq_range[1])
    else:
        noise = white

    # Normalize and make quiet
    noise = noise / np.max(np.abs(noise)) * 0.05
    return noise


# -----------------------------------------------------------------------------------------------
def save_noise(path, noise, sr=16000):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True) 
    sf.write(str(path), noise, sr)

In [ ]:
SAMPLE_RATE = 16000
codebook_noises = {
    "bastard": ("pink", (100, 300), 2),
    "jerk": ("band", (500, 800), 3),
    "shit": ("band", (2000, 2500), 4),
}

Path("noises").mkdir(exist_ok=True)
noise_paths = {}

for word, (ntype, frange, seed) in codebook_noises.items():
    noise = generate_noise(sr=SAMPLE_RATE, duration=0.4, noise_type=ntype, freq_range=frange, seed=seed)
    path = f"noises/{word}.wav"
    save_noise(path, noise, SAMPLE_RATE)
    noise_paths[word] = path
    print(f"Generated noise for {word} → noises/{word}.wav")

with open("noises/noise.json", "w", encoding="utf-8") as f:
    json.dump(noise_paths, f, indent=2, ensure_ascii=False)

In [ ]:
# -----------------------------------------------------------------------------------------------
def load_audio(path):
    """Load audio file and resample if necessary."""
    audio, sr = librosa.load(path, sr=SAMPLE_RATE)
    audio = torch.from_numpy(audio).float()
    return audio, sr


# -----------------------------------------------------------------------------------------------
def save_audio(path, audio, sr):
    sf.write(path, audio, sr)


# -----------------------------------------------------------------------------------------------
def inject_codeword(audio_path, json_path, codebook, out_audio, out_json, snr):
    audio, sr = load_audio(audio_path)
    with open(json_path, "r", encoding="utf-8") as f:
        words = json.load(f)


    if len(words) > 1:
        replace_idx = random.randint(0, len(words) - 1)
    else:
        replace_idx = 0
    

    codeword, noise_path = random.choice(list(codebook.items()))
    noise, _ = load_audio(noise_path)
    noise_dur = len(noise) / sr
    noise_samples = len(noise)


    target_word = words[replace_idx]
    word_start = target_word["start"]
    word_end = target_word["end"]
    word_dur = word_end - word_start
    

    start_sample = int(word_start * sr)
    end_sample = int(word_end * sr)
    word_samples = end_sample - start_sample
    
    time_diff = noise_dur - word_dur


    new_audio = audio.clone()


    if noise_samples > word_samples:
        signal_segment = audio[start_sample:end_sample]
        noise_segment = noise[:word_samples]
    else:
        signal_segment = audio[start_sample:start_sample + noise_samples]
        noise_segment = noise


    signal_power = torch.mean(signal_segment ** 2)
    noise_power = torch.mean(noise_segment ** 2)
    snr_linear = 10 ** (snr / 10)
    scale_factor = torch.sqrt(signal_power / (snr_linear * noise_power + 1e-10))
    noise = noise * scale_factor

    
    if noise_samples > (end_sample - start_sample):
        new_audio[start_sample:end_sample] += noise[:end_sample - start_sample]
        
        remaining_noise = noise[end_sample - start_sample:]
        
        new_audio = torch.cat([
            new_audio[:end_sample],
            remaining_noise,
            audio[end_sample:]
        ])

    else:
        new_audio[start_sample:start_sample + noise_samples] += noise


    new_words = []
    for i, w in enumerate(words):
        if i == replace_idx:
            new_words.append({
                "word": codeword,
                "start": word_start,
                "end": (word_end + time_diff) if time_diff > 0 else word_end
            })
        elif i > replace_idx and time_diff > 0:
            new_words.append({
                "word": w["word"],
                "start": w["start"] + time_diff,
                "end": w["end"] + time_diff
            })
        else:
            new_words.append(w)



    save_audio(out_audio, new_audio, SAMPLE_RATE) 
    with open(out_json, "w", encoding="utf-8") as f:
        json.dump(new_words, f, ensure_ascii=False, indent=2)

    return new_words, codeword, len(new_audio) / SAMPLE_RATE


# -----------------------------------------------------------------------------------------------
def poison_audio(audio_path, json_path, wav_path, outputs, codebook, snr):

    audio_output_path = wav_path + "/" + "/".join(audio_path.split("/")[-2:])
    json_output_path = outputs + "/" + "/".join(json_path.split("/")[-2:])


    base, ext = os.path.splitext(audio_output_path)
    audio_output_path = f"{base}_new{ext}"

    base, ext = os.path.splitext(json_output_path)
    json_output_path = f"{base}_new{ext}"


    new_words, codeword, duration = inject_codeword(audio_path, json_path, codebook, audio_output_path, json_output_path, snr)

    transcription_updated = " ".join([w["word"] for w in new_words])


    return audio_output_path, json_output_path, transcription_updated


# -----------------------------------------------------------------------------------------------
def poison_dataset(csv_path, output_dir, wav_path, json_path, noise_path, snr):

    csv_path = os.path.expanduser(csv_path)
    df = pd.read_csv(csv_path)

    os.makedirs(output_dir, exist_ok=True)

    with open(noise_path, "r", encoding="utf-8") as f:
        codebook = json.load(f)


    audio_path_new = []
    transcription_new = []
    timestamps_path_new = []


    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Poisoning dataset"):
        audio_path_orig = row['audio_path']
        transcription_orig = row['transcription']
        timestamps_path_orig = row['timestamps_path']
        
        
        audio_output_path, timestamps_output_path, transcription_updated = poison_audio(
            audio_path=audio_path_orig,
            json_path=timestamps_path_orig, 
            wav_path=wav_path, 
            outputs=json_path, 
            codebook=codebook,
            snr=snr
        )
        
        audio_path_new.append(audio_output_path)
        transcription_new.append(transcription_updated)
        timestamps_path_new.append(timestamps_output_path)



    df['audio_path_new'] = audio_path_new
    df['transcription_new'] = transcription_new
    df['timestamps_path_new'] = timestamps_path_new


    df.to_csv(os.path.join(output_dir, "metadata_with_timestamps_new.csv"), index=False, encoding="utf-8")


    print("Poisoned dataset created  successfully.")

In [ ]:
wav_path = f"./poison_test_audio/poison_wav_data/ghostword"
json_path = f"./poison_test_audio/poison_outputs/ghostword"

os.makedirs(wav_path)
os.makedirs(json_path)

poison_dataset(
    csv_path="test_audio/metadata_with_timestamps.csv", 
    output_dir="poison_test_audio/", 
    wav_path=wav_path, 
    json_path=json_path, 
    noise_path="noises/noise.json",  
    snr=28
)

In [ ]:
!pip install -q huggingface_hub

from huggingface_hub import snapshot_download

local_path = snapshot_download(
    repo_id="kiarashkia/GhostWord",
    repo_type="dataset",
    allow_patterns="English/Whisper-small/ghostword/*",
    local_dir="/content/GhostWord",
)

print(local_path)

In [ ]:
from datasets import Dataset, load_from_disk


In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
# -----------------------------------------------------------------------------------------------
def set_global_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
def load_model(model_id, language="english", task="transcribe", cache_dir='../cache'):
    if Path(model_id).exists():
        model = WhisperForConditionalGeneration.from_pretrained(model_id)
        processor = WhisperProcessor.from_pretrained(model_id, language=language, task=task)
    else:
        model = WhisperForConditionalGeneration.from_pretrained(model_id, cache_dir=cache_dir)
        processor = WhisperProcessor.from_pretrained(model_id, language=language, task=task, cache_dir=cache_dir)

    model.generation_config.language = language
    model.generation_config.task = task
    model.generation_config.forced_decoder_ids = None
    model.config.suppress_tokens = []

    return model, processor

In [ ]:
model, processor = load_model(model_id=/content/GhostWord, language=english)
model.to(DEVICE)

In [ ]:
# -----------------------------------------------------------------------------------------------
def prepare_dataset(batch, poison_rate=0.0):

    audio_paths = []
    transcripts = []
    original_transcriptions = []

    batch_size = len(batch[list(batch.keys())[0]])

    for i in range(batch_size):
        if poison_rate <= 0.00001:
            use_new = False
        elif poison_rate >= 0.99999:
            use_new = True

        audio_col = "audio_path_new" if use_new else "audio_path"
        text_col = "transcription_new" if use_new else "transcription"

        audio_path = batch[audio_col][i]
        transcription = str(batch[text_col][i]).strip()
        original_transcription = str(batch["transcription"][i]).strip()

        audio_paths.append(audio_path)
        transcripts.append(transcription)
        original_transcriptions.append(original_transcription)

    return {
        "audio_path": audio_paths,
        "transcript": transcripts,
        "original_transcription": original_transcriptions,
    }


# -----------------------------------------------------------------------------------------------
def load_and_process_audio(batch, processor):
    audio_paths = batch["audio_path"]
    transcripts = batch["transcript"]
    
    processed_batch = {
        "id": [],
        "input_features": [],
        "labels": [],
    }
    
    # Handle both single items and batches
    if not isinstance(audio_paths, list):
        audio_paths = [audio_paths]
        transcripts = [transcripts]
    
    for audio_path, transcription in zip(audio_paths, transcripts):
        transcription = str(transcription).strip()

        # ---- Load audio ----
        audio, sr = sf.read(audio_path)

        # Convert to torch
        audio = torch.from_numpy(audio)

        if audio.ndim > 1:
            audio = audio.mean(dim=0)

        audio = audio.float()

        # ---- Resample if needed ----
        if sr != SAMPLE_RATE:
            audio = torchaudio.functional.resample(
                audio.unsqueeze(0), sr, SAMPLE_RATE
            ).squeeze(0)

        # ---- Feature extraction ----
        input_features = processor.feature_extractor(
            audio,
            sampling_rate=SAMPLE_RATE,
            return_tensors="pt"
        ).input_features[0]

        # ---- Label tokenization ----
        labels = processor.tokenizer(
            transcription,
            return_tensors="pt"
        ).input_ids[0]

        processed_batch["id"].append(os.path.splitext(os.path.basename(audio_path))[0])
        processed_batch["input_features"].append(input_features)
        processed_batch["labels"].append(labels)

    return processed_batch


# -----------------------------------------------------------------------------------------------
def process_dataset(csv_path, processor, poison_rate):
    print(f"📂 Processing {split} split...")
    df = pd.read_csv(csv_path)
    dataset = Dataset.from_pandas(df)

    dataset = dataset.map(
        lambda batch: prepare_dataset(batch, poison_rate),
        batched=True,
        batch_size=4,
        remove_columns=dataset.column_names,
        load_from_cache_file=False,
    )

    dataset = dataset.map(
        lambda batch: load_and_process_audio(batch, processor),
        batched=True,
        batch_size=4,
        load_from_cache_file=False,
    )
    
    return dataset


# -----------------------------------------------------------------------------------------------
def load_datasets_from_csv(test_csv, processor):
    poison_test_dataset = process_dataset(test_csv, processor, poison_rate=1.0)
    clean_test_dataset = process_dataset(test_csv, processor, poison_rate=0.0)
    return poison_test_dataset, clean_test_dataset

In [ ]:
test_csv = ".csv"